# <center><font size=10>**Natural Language Processing with Generative AI: Medical Assistant**</font></center>
---

## <span style="color:#87CEEB;">**Problem Statement**</span>

### Business Context

The healthcare sector faces growing challenges in managing large volumes of medical information while maintaining diagnostic speed and accuracy. Clinicians often experience information overload, making it difficult to access reliable, up‑to‑date knowledge when making critical decisions. Streamlined, centralized systems that provide rapid access to trusted medical resources can significantly enhance diagnostic efficiency, support informed decision‑making, and improve overall patient care.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

- **Understand** the challenge of information overload faced by healthcare professionals when accessing vast medical knowledge.  
- **Apply** Retrieval‑Augmented Generation (RAG) techniques to deliver fast, reliable, and context‑aware medical information.  
- **Analyze** how AI‑driven knowledge retrieval can improve diagnostic accuracy and patient outcomes.  
- **Evaluate** the potential of AI systems to standardize clinical decision‑making and care practices.  
- **Create** a functional RAG prototype using trusted medical manuals to demonstrate feasibility and real‑world effectiveness.

### Data Description

The [**Merck Manuals**](medical_diagnosis_manual.pdf) are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

---
## <span style="color:#87CEEB;">**Installing and Importing Necessary Libraries and Dependencies**</span>

In [ ]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
# !CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

# Installation for CPU llama-cpp-python 
# set CMAKE_ARGS=-DLLAMA_CUBLAS=off
# set FORCE_CMAKE=1
# pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir
# pip install requirements.txt --force-reinstall --no-cache-dir -q

In [3]:
#Libraries for processing dataframes,text
import pandas as pd
import time
import re
import json
import os

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_community.vectorstores import FAISS


# Libraries for evaluation
from difflib import SequenceMatcher

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

# For cleaning the manual text
from copy import deepcopy


---
## <span style="color:#87CEEB;">**Utility Functions**</span>

In [2]:
# Function to get response from the model with timing

def get_response(
    llm,
    prompt: str,
    max_tokens: int = 128,
    temperature: float = 0,
    top_p: float = 0.95,
    top_k: int = 50,
    stop=None
):
    """
    Generate a response with:
        - KV cache reset
        - hard max_tokens limit
        - safe stop sequence
        - timing and length stats

    Returns:
        dict with:
            - "text": generated answer
            - "time": seconds elapsed
            - "chars": character length
            - "tokens": approximate token count (whitespace split)
    """

    if stop is None:
        stop = ["</s>", "\n\n\n"]  # safe default

    llm.reset() #

    start = time.time()

    output = llm(
        prompt=prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        stream=False,
        stop=stop
    )

    elapsed = time.time() - start
    text = output["choices"][0]["text"].strip()

    stats = {
        "text": text,
        "time": elapsed,
        "chars": len(text),
        "tokens": len(text.split())
    }

    return stats


In [3]:
# Function to run the baseline experiment with default parameters

def run_baseline(llm, questions):
    """
    Baseline experiment with default parameters.
    Uses get_response() which returns a stats dictionary.
    Output format matches the structure of other experiment runs.
    """

    print("\n========== BASELINE (DEFAULT PARAMETERS) ==========\n")
    results = []

    for idx, q in enumerate(questions, start=1):
        print(f"\n--- BASELINE QUESTION {idx} of {len(questions)} ---\n")
        print(f"Q: {q}\n")

        stats = get_response(llm, q)   # returns dict with text, time, chars, tokens, truncated flag

        # Detect truncation (if your get_response returns a flag)
        truncated = stats.get("truncated", False)
        completion_status = "Completed" if not truncated else "Truncated"

        print("ANSWER:\n")
        print(stats["text"])
        print(f"\nTime taken: {stats['time']:.2f} seconds")
        print(f"Chars: {stats['chars']} | Tokens: {stats['tokens']}")
        print(f"Status: {completion_status}")
        print("\n---------------------------------------------\n")

        results.append({
            "question": q,
            "answer": stats["text"],
            "time": stats["time"],
            "chars": stats["chars"],
            "tokens": stats["tokens"],
            "truncated": truncated,
            "completed": not truncated
        })

    return results


In [4]:
# Function to build a prompt in the instruction format expected by Mistral-Instruct models

def build_mistral_instruction_prompt(instruction: str, question: str):
    """
    Build a prompt using the instruction format expected by Mistral-Instruct models.
    This usually improves instruction-following and structure.
    """
    return f"<s>[INST] {instruction}\n\nQuestion: {question} [/INST]"

In [5]:
# Function to run all prompt-engineering experiments on a list of questions

def run_all_experiments(llm, questions, experiment_configs, save_path="prompt_experiment_results.json"):
    """
    Run all prompt experiments with:
    - Mistral instruction formatting
    - timing and length stats
    - checkpoint saving after every single run
    """

    # Load prior results if they exist, so you can resume instead of restarting
    if os.path.exists(save_path):
        with open(save_path, "r", encoding="utf-8") as f:
            all_results = json.load(f)
    else:
        all_results = {}

    for exp in experiment_configs:
        name = exp["name"]
        instruction = exp["instruction"]

        print(f"\n========== {name.upper()} ==========\n")

        if name not in all_results:
            all_results[name] = []

        completed_questions = {r["question"] for r in all_results[name]}

        for idx, q in enumerate(questions, start=1):
            if q in completed_questions:
                print(f"Skipping already completed: {name} | Question {idx}")
                continue

            full_prompt = build_mistral_instruction_prompt(instruction, q)

            print(f"\n--- {name} QUESTION {idx} of {len(questions)} ---\n")
            print(f"Q: {q}\n")

            stats = get_response(
                llm,
                full_prompt,
                max_tokens=exp["max_tokens"],
                temperature=exp["temperature"],
                top_p=exp["top_p"],
                top_k=exp["top_k"],
                stop=["</s>", "\n\n\n"]
            )

            print("ANSWER:\n")
            print(stats["text"])
            print(f"\nTime: {stats['time']:.2f}s | Chars: {stats['chars']} | Tokens: {stats['tokens']}")
            print("\n---------------------------------------------\n")

            all_results[name].append({
                "question": q,
                "answer": stats["text"],
                "time": stats["time"],
                "chars": stats["chars"],
                "tokens": stats["tokens"]
            })

            # Save checkpoint after each run
            with open(save_path, "w", encoding="utf-8") as f:
                json.dump(all_results, f, indent=2, ensure_ascii=False)

    return all_results

In [6]:
# Function to build a comparison table from baseline and experiment results

def build_comparison_table(baseline_results, experiment_results):
    """
    Build a comparison table comparing baseline vs prompt-engineering experiments.

    Metrics included:
    - Avg / Min / Max time
    - Clean ending count (proxy only, not true answer completeness)
    - Possible truncations
    - Avg chars
    - Avg tokens
    - Delta vs baseline
    - Rankings

    Notes:
    - 'Ended Cleanly' means the answer ended with '.', '!' or '?'
    - This is only a surface-level proxy for completion
    - 'Possible Truncations' flags responses that did not end cleanly
    """

    def avg(values):
        return sum(values) / len(values) if values else 0

    def extract(field, results):
        return [r[field] for r in results]

    def ended_cleanly_stats(results):
        endings = (".", "!", "?")
        clean = sum(1 for r in results if r["answer"].strip().endswith(endings))
        total = len(results)
        return clean, total

    def possible_truncation_count(results):
        endings = (".", "!", "?")
        return sum(1 for r in results if not r["answer"].strip().endswith(endings))

    rows = []

    # -------------------------
    # Baseline metrics
    # -------------------------
    base_avg_time = avg(extract("time", baseline_results))
    base_avg_chars = avg(extract("chars", baseline_results))
    base_avg_tokens = avg(extract("tokens", baseline_results))
    base_clean, base_total = ended_cleanly_stats(baseline_results)
    base_trunc = possible_truncation_count(baseline_results)

    rows.append({
        "Experiment": "Baseline (Default)",
        "Avg Time (s)": base_avg_time,
        "Min Time (s)": min(extract("time", baseline_results)),
        "Max Time (s)": max(extract("time", baseline_results)),
        "Ended Cleanly": f"{base_clean}/{base_total}",
        "Possible Truncations": base_trunc,
        "Avg Chars": base_avg_chars,
        "Avg Tokens": base_avg_tokens,
        "Δ Time vs Baseline": 0.0,
        "Δ Chars vs Baseline": 0.0,
        "Δ Tokens vs Baseline": 0.0,
    })

    # -------------------------
    # Experiment rows
    # -------------------------
    for name, results in experiment_results.items():
        avg_time = avg(extract("time", results))
        avg_chars = avg(extract("chars", results))
        avg_tokens = avg(extract("tokens", results))
        clean, total = ended_cleanly_stats(results)
        trunc = possible_truncation_count(results)

        rows.append({
            "Experiment": name,
            "Avg Time (s)": avg_time,
            "Min Time (s)": min(extract("time", results)),
            "Max Time (s)": max(extract("time", results)),
            "Ended Cleanly": f"{clean}/{total}",
            "Possible Truncations": trunc,
            "Avg Chars": avg_chars,
            "Avg Tokens": avg_tokens,
            "Δ Time vs Baseline": avg_time - base_avg_time,
            "Δ Chars vs Baseline": avg_chars - base_avg_chars,
            "Δ Tokens vs Baseline": avg_tokens - base_avg_tokens,
        })

    df = pd.DataFrame(rows)

    # -------------------------
    # Rankings
    # -------------------------
    # Lower average time = better
    df["Rank: Fastest"] = df["Avg Time (s)"].rank(method="min", ascending=True)

    # Lower average chars = shorter / more concise
    df["Rank: Most Concise"] = df["Avg Chars"].rank(method="min", ascending=True)

    # Fewer possible truncations = better
    df["Rank: Least Truncated"] = df["Possible Truncations"].rank(method="min", ascending=True)

    # Round float columns
    float_cols = df.select_dtypes(include="float").columns
    df[float_cols] = df[float_cols].round(2)

    # Optional: sort with baseline first, then fastest experiment
    baseline_df = df[df["Experiment"] == "Baseline (Default)"]
    experiments_df = df[df["Experiment"] != "Baseline (Default)"].sort_values("Avg Time (s)")
    df = pd.concat([baseline_df, experiments_df], ignore_index=True)

    return df

---
## <span style="color:#87CEEB;">**Question Answering Using LLM (Test Question & Reponse)**</span>

### Downloading and Loading a test model

In [8]:
# Define the test model details
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_basename = "mistral-7b-instruct-v0.2.Q6_K.gguf"

In [9]:
# Download the model from Hugging Face Hub and save it to a path
model_path = hf_hub_download(
    repo_id= model_name_or_path,
    filename= model_basename,
    local_dir="models"
)

model_path

'models\\mistral-7b-instruct-v0.2.Q6_K.gguf'

In [10]:
# Test loading the model 

print("Loading model...")

llm = Llama(
    model_path=model_path,
    n_ctx=1024,              # smaller context = faster
    n_cores=-2,
    n_batch=512,             # to improve throughput
    verbose=True,            # avoid Jupyter fileno() suppression error
    logits_all=False,        # only final output logits
    embedding=False          # generation only, not embedding mode
)

print("Model loaded successfully.\n")


Loading model...
Model loaded successfully.



AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 0 | SSE3 = 1 | SSSE3 = 0 | VSX = 0 | 


### Test Response

In [11]:
response = get_response(llm, "What treatment options are available for managing hypertension?")
response

{'text': 'Hypertension, or high blood pressure, is a common condition that can increase the risk of various health problems such as heart disease, stroke, and kidney damage. The good news is that there are several effective treatment options available to help manage hypertension and reduce the risk of complications. Here are some of the most commonly used treatments:\n\n1. Lifestyle modifications: Making lifestyle changes is often the first line of defense against hypertension. This may include eating a healthy diet rich in fruits, vegetables, whole grains, and lean proteins; limiting sodium intake; getting regular physical activity',
 'time': 107.56607222557068,
 'chars': 628,
 'tokens': 95}

---
## <span style="color:#87CEEB;">**Baseline QA with instruction-tuned LLM (Project Questions & Response)**</span>

In [12]:
# convert all questions into a list for easy access and processing
questions = [
    "What is the protocol for managing sepsis in a critical care unit?",
    "What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?",
    "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?",
    "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?",
    "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
]


### Baseline Response (no prompt engineering)
- This captures the model’s natural behaviour. It shows how the model answers without structure, guidance, or constraints and gives a reference point to compare the engineered prompts against.

In [13]:
#  baseline experiment with no prompt engineering, just the question as input to the model and recording the response for each question.

baseline_results = run_baseline(llm, questions)


========== BASELINE (DEFAULT PARAMETERS) ==========


--- BASELINE QUESTION 1 of 5 ---

Q: What is the protocol for managing sepsis in a critical care unit?

ANSWER:

Sepsis is a life-threatening condition that can arise from an infection, and it requires prompt recognition and aggressive management in a critical care unit. The following are general steps for managing sepsis in a critical care unit:

1. Early recognition and suspicion: Septic patients may present with non-specific symptoms such as fever, chills, tachycardia, tachypnea, altered mental status, or lactic acidosis. It is essential to have a high index of suspicion for sepsis in patients with suspected or confirmed infection.
2. Early goal-directed

Time taken: 78.83 seconds
Chars: 552 | Tokens: 83
Status: Completed

---------------------------------------------


--- BASELINE QUESTION 2 of 5 ---

Q: What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be fo

**Observation:**

- The baseline experimentused the pretrained Mistral-7B-Instruct-v0.2 model in a direct question-answering setting without document retrieval. The model produced generally relevant and medically plausible answers across all five test questions, demonstrating useful pretrained medical knowledge. However, the responses remained generic and were not grounded in the Merck Manual, which limits trustworthiness in a healthcare setting with response times ranging from approximately 65–75seconds per query.  In addition, several answers were truncated mid-sentence or mid-list, indicating that the baseline token limit was still insufficient for more detailed clinical responses. Overall, the settings did not address the key limitations of completeness and source grounding, reinforcing the need for prompt engineering and a FAISS-based RAG pipeline.

---
## <span style="color:#87CEEB;">**Baseline QA with instruction-tuned LLM with Prompt Engineering (Project Questions & Response)**</span>

Prompt engineering was tested using the same Mistral-7B-Instruct model, with decoding parameters largely fixed to the baseline for fairness; only max_tokens was increased where necessary to prevent truncation in more structured response formats.

**Purpose:**
Test whether better prompting alone improves clarity, structure, and completeness before introducing retrieval with FAISS.

### LLM prompt Parameters

In [24]:
# Prompt engineering experiments to test different instruction styles and output formats.

# Note:
# Prompt engineering experiments use the same Mistral-7B-Instruct model.
# The model itself is not fine-tuned; only the instruction prompts are modified.
# Prompts are formatted using the Mistral instruction format (<s>[INST] ... [/INST]) to improve instruction-following and output structure.

# Fair comparison logic:
# - temperature, top_p, and top_k are kept the same as baseline in all modes
#   so that generation behaviour stays as close as possible to the original setup.
# - This means any improvement is mainly due to prompt design, not sampling changes.
# - max_tokens is increased only where needed to reduce truncation caused by
#   longer requested output structures (bullets, sections, expert summaries).

# Baseline reference:
#   max_tokens = 128
#   temperature = 0.0
#   top_p = 0.95
#   top_k = 50

experiment_configs = [
    {
        "name": "Clinical Guideline Mode",
        # Rationale:
        # Short bullet points are easy to compare and naturally reduce output length.
        # "Exactly 4" prevents the model from expanding to 6–8 bullets.
        # "No intro or conclusion" removes wasted tokens.
        "instruction": (
            "You are a medical knowledge assistant. "
            "Answer in exactly 4 short bullet points. "
            "Each bullet must be one complete sentence. "
            "Focus on key clinical guidance only. "
            "No introduction and no conclusion."
        ),
        # Params kept same as baseline for fairness:
        # - temperature=0.0 keeps outputs deterministic
        # - top_p=0.95 and top_k=50 remain unchanged to isolate prompt effects
        # Param changed:
        # - max_tokens reduced to 96 to speed up generation and enforce brevity
        "temperature": 0.0,
        "top_p": 0.95,
        "top_k": 50,
        "max_tokens": 96
    },
    {
        "name": "Explain Like I'm 15",
        # Rationale:
        # This tests plain-language explanation.
        # Limiting to exactly 3 short sentences keeps output fast and controlled.
        "instruction": (
            "You are a medical knowledge assistant. "
            "Explain this in exactly 3 short sentences for a 15-year-old. "
            "Avoid jargon. Keep the meaning medically correct. "
            "No introduction and no conclusion."
        ),
        # Only max_tokens is kept low because this mode should be very short.
        "temperature": 0.0,
        "top_p": 0.95,
        "top_k": 50,
        "max_tokens": 96
    },
    {
        "name": "Structured Medical Framework",
        # Rationale:
        # This is the most evaluation-friendly format.
        # One sentence per section prevents section sprawl.
        "instruction": (
            "You are a medical knowledge assistant. "
            "Answer using exactly these 4 sections: "
            "1. Definition "
            "2. Key symptoms "
            "3. Causes "
            "4. General management principles. "
            "Write exactly 1 short sentence per section. "
            "No introduction and no conclusion."
        ),
        # max_tokens slightly higher because 4 labelled sections need extra tokens.
        "temperature": 0.0,
        "top_p": 0.95,
        "top_k": 50,
        "max_tokens": 120
    },
    {
        "name": "Expert Deep-Dive Mode",
        # Rationale:
        # This keeps an expert-style answer but cuts it down hard.
        # Exactly 5 concise sentences is far cheaper than open-ended deep dives.
        "instruction": (
            "You are a medical knowledge assistant. "
            "Provide an expert-level answer in exactly 5 concise sentences. "
            "Prioritise mechanism, key clinical features, and management principles. "
            "Do not add filler."
        ),
        # This mode needs a bit more room, but still much less than before.
        "temperature": 0.0,
        "top_p": 0.95,
        "top_k": 50,
        "max_tokens": 144
    },
    {
        "name": "Bullet-Point Precision Mode",
        # Rationale:
        # This is a compact factual mode.
        # Keeping it at exactly 4 bullets reduces variability and runtime.
        "instruction": (
            "You are a medical knowledge assistant. "
            "Answer in exactly 4 bullet points. "
            "Each bullet must contain one important clinical fact in one sentence. "
            "Do not repeat ideas."
        ),
        "temperature": 0.0,
        "top_p": 0.95,
        "top_k": 50,
        "max_tokens": 96
    }
]

In [29]:
experiment_results = run_all_experiments(llm, questions, experiment_configs)


========== CLINICAL GUIDELINE MODE ==========


--- Clinical Guideline Mode QUESTION 1 of 5 ---

Q: What is the protocol for managing sepsis in a critical care unit?

ANSWER:

1. Immediate resuscitation with intravenous fluids and oxygen to maintain adequate tissue perfusion and oxygenation.
2. Administration of broad-spectrum antibiotics as soon as possible based on suspected infection source and local microbiology data.
3. Close monitoring of vital signs, lactate levels, urine output, and organ function with frequent reassessment and adjustment of treatment plan accordingly.
4. Consideration of vasopress

Time: 79.81s | Chars: 437 | Tokens: 59

---------------------------------------------


--- Clinical Guideline Mode QUESTION 2 of 5 ---

Q: What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

ANSWER:

1. Appendicitis is characterized by abdominal pain, usually located in the lower right qua

**Observation**

- The prompt-engineering comparison focused on output efficiency and structural stability rather than factual grounding. Since these experiments were performed without retrieval, the table evaluates response time, output length, and whether responses appeared to end cleanly. This helps identify which prompting strategy produces the most stable and concise answers before introducing the FAISS-based RAG pipeline.

- **The prompt-engineering experiments showed that the same Mistral-7B-Instruct model could produce noticeably better structured and more readable answers when guided with carefully designed prompts.** 
- Among the tested configurations,**Structured Medical Framework** provided the most consistent output organization, while **Explain Like I’m 15 and Bullet-Point Precision Mode** were generally faster and more concise. 
- **Expert Deep-Dive Mode produced the richest responses but at higher runtime cost**.
- However, despite improvements in formatting and task alignment, several outputs across different modes still appeared truncated, and all answers remained ungrounded in the Merck Manual. This confirms that prompt engineering alone improves presentation, but a FAISS-based RAG pipeline is still required to improve factual grounding and trustworthiness.

### Experiment Results Interpreted Against Baseline

In [30]:
# Build and print the comparison table summarising all results

comparison_table = build_comparison_table(baseline_results, experiment_results)
comparison_table


,Experiment,Avg Time (s),Min Time (s),Max Time (s),Ended Cleanly,Possible Truncations,Avg Chars,Avg Tokens,Δ Time vs Baseline,Δ Chars vs Baseline,Δ Tokens vs Baseline,Rank: Fastest,Rank: Most Concise,Rank: Least Truncated
0,Baseline (Default),85.97,78.83,95.79,1/5,4,537.2,86.8,0.00,0.0,0.0,3.0,5.0,2.0
1,Bullet-Point Precision Mode,73.39,69.09,75.89,1/5,4,421.6,60.8,-12.59,-115.6,-26.0,1.0,2.0,2.0
2,Clinical Guideline Mode,83.67,79.81,88.91,1/5,4,427.0,59.6,-2.31,-110.2,-27.2,2.0,3.0,2.0
3,Explain Like I'm 15,86.51,74.49,92.35,1/5,4,416.8,67.4,0.54,-120.4,-19.4,4.0,1.0,2.0
4,Structured Medical Framework,91.75,84.04,99.89,4/5,1,477.8,66.4,5.77,-59.4,-20.4,5.0,4.0,1.0
5,Expert Deep-Dive Mode,105.64,101.38,114.29,1/5,4,591.2,83.4,19.67,54.0,-3.4,6.0,6.0,2.0


**Report Summary Table**

| Insight Category                | Observation vs Baseline                                                                                                                                           | Interpretation                                                                                                            | Implication for Project                                                                        |
| ------------------------------- | ----------------------------------------------------------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------- | ---------------------------------------------------------------------------------------------- |
| Response Time                   | Most prompt modes were slightly slower than baseline, especially Structured and Expert modes. Bullet-Point and Explain Like I'm 15 were closer to baseline speed. | More structured prompts require the model to generate formatted and organised responses, which increases generation time. | There is a trade-off between response quality and speed when using structured prompts.         |
| Output Length                   | Expert and Structured modes produced longer responses than baseline, while Bullet-Point and Clinical modes produced shorter responses.                            | Prompt instructions strongly influenced verbosity and structure of the answers.                                           | Prompt engineering can control response length and format without changing the model.          |
| Structure & Formatting          | All prompt modes produced more structured answers than baseline, especially Structured Medical Framework and Clinical Guideline Mode.                             | Prompt engineering significantly improves answer organisation even without retrieval.                                     | Structured prompting improves readability and usability of model outputs.                      |
| Readability                     | Explain Like I’m 15 produced simpler and more readable answers compared to baseline and expert modes.                                                             | Simpler prompts can improve clarity for non-technical users.                                                              | Prompt design can adapt the same model for different audiences.                                |
| Detail Level                    | Expert Deep-Dive Mode produced much more detailed answers than baseline.                                                                                          | Prompt instructions directly control level of detail generated by the model.                                              | Prompt engineering allows control over response depth without changing the model architecture. |
| Truncation / Incomplete Answers | Truncation occurred across baseline and most experiment modes, especially longer structured prompts.                                                              | Increasing structure or detail increases risk of output truncation due to token limits.                                   | Token limits are a major constraint and must be managed before RAG implementation.             |
| Consistency Across Questions    | Structured Medical Framework produced the most consistent answer format across all 5 questions compared to baseline.                                              | Structured prompts improve consistency and comparability of responses.                                                    | Consistent output structure is useful for evaluation and downstream processing.                |
| Overall Quality vs Baseline     | All prompt-engineered modes produced better formatted and more task-aligned answers than baseline.                                                                | Prompt engineering improves output quality without modifying the model.                                                   | Prompt engineering is an important step before implementing RAG.                               |
| Grounding / Source Accuracy     | None of the prompt modes were grounded in Merck Manual documents, same as baseline.                                                                               | Prompt engineering improves format but does not ensure factual grounding.                                                 | Justifies the need for a RAG pipeline.                                                         |


---
## <span style="color:#87CEEB;">**RAG Pipeline Implementation → FAISS → Retrieval → Augmented Prompt → Answer Generation.**</span>
- This stage converts the raw medical manual into structured, searchable knowledge that can be retrieved and used by the language model to generate grounded medical answers. RAG improves LLM responses by retrieving relevant information from a document database and using that information as context before generating the final answer.

### RAG Pipeline Workflow & Summary
- The Retrieval-Augmented Generation (RAG) pipeline improves the quality and reliability of model responses by retrieving relevant information from the Merck medical manual before generating an answer. The document is first converted into text, split into smaller chunks, embedded into vector representations, and stored in a FAISS vector database. When a user asks a question, the system retrieves the most relevant document chunks using similarity search and provides this context to the language model, which then generates a response based on the retrieved medical information.

                  Merck Manual PDF  
                  ↓  
                  **Text Extraction**  
                  ↓  
                  **Text Chunking**  
                  ↓  
                  **Generate Embeddings**  
                  ↓  
                  **Store in FAISS Vector Index**  
                  ↓  
                  **User Question**  
                  ↓  
                  **Embed Question**  
                  ↓  
                  **Retrieve Top‑K Relevant Chunks**  
                  ↓  
                  **Send Retrieved Context + Question to LLM**  
                  ↓  
                  **Generate Final Answer**


**RAG Pipeline Summary**

| Step                 | Description                                  | Purpose                                                  |
| -------------------- | -------------------------------------------- | -------------------------------------------------------- |
| Document Loading     | Medical manual PDF loaded and text extracted | Converts document into machine-readable text             |
| Chunking             | Text split into overlapping chunks           | Ensures text fits embedding limits and preserves context |
| Embeddings           | Each chunk converted into vector embeddings  | Enables semantic similarity search                       |
| Vector Store         | Embeddings stored in FAISS index             | Allows fast retrieval of relevant document sections      |
| Query Embedding      | User question converted into embedding       | Makes question searchable in vector space                |
| Retrieval            | Top-k most similar chunks retrieved          | Provides relevant medical context                        |
| Augmented Generation | Retrieved context + question sent to LLM     | Generates grounded medical answer                        |

### Loading the Data

In [7]:
manual_pdf_path = "medical_diagnosis_manual.pdf" 
pdf_loader = PyMuPDFLoader(manual_pdf_path)
manual = pdf_loader.load()

### Data Overview

-  Checking the first 5 pages and manual length

In [8]:
for i in range(5):
    print(f"Page Number : {i+1}",end="\n")
    print(manual[i].page_content,end="\n")
    
print(f"\nLength of manual: {len(manual)}")

Page Number : 1
omotayo@outlook.com
EGVD5P09O3
This file is meant for personal use by omotayo@outlook.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 2
omotayo@outlook.com
EGVD5P09O3
This file is meant for personal use by omotayo@outlook.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 3
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    .......................................................................................................................................

### Data Cleaning and Normalisation

- The extracted PDF text contained watermarks, table-of-contents entries, page numbers, and formatting artefacts that are not useful for retrieval. 
- A light text-cleaning step was applied before chunking to remove repeated watermark text, legal notices, email addresses, standalone page numbers, and formatting artefacts introduced during PDF extraction. *The cleaning was intentionally conservative so that medically relevant content, chapter headings, and page structure were preserved for downstream chunking and retrieval.*

In [9]:
# Utility function to clean the manual text by removing noise and irrelevant content while preserving important medical information for better processing in the RAG pipeline.

def clean_manual_docs(documents):
    """
    Light cleaning for RAG preparation.

    Keeps most page content intact and removes only obvious noise:
    - watermark/legal notice lines
    - email addresses
    - ID-like codes
    - standalone page numbers
    - hyphenated line breaks
    - excess whitespace
    """

    cleaned_docs = []

    for doc in documents:
        # avoid mutating the original docs
        new_doc = deepcopy(doc)
        text = new_doc.page_content

        # Remove email addresses
        text = re.sub(r"\b\S+@\S+\b", " ", text, flags=re.IGNORECASE)

        # Remove short uppercase/alphanumeric watermark codes like EGVD5P09O3
        text = re.sub(r"\b[A-Z0-9]{8,}\b", " ", text)

        # Remove watermark / legal notice lines
        text = re.sub(r"(?im)^.*personal use.*$", " ", text)
        text = re.sub(r"(?im)^.*sharing or publishing.*$", " ", text)
        text = re.sub(r"(?im)^.*legal action.*$", " ", text)

        # Remove standalone page numbers only
        text = re.sub(r"(?m)^\s*\d+\s*$", " ", text)

        # Fix hyphenated line breaks: e.g. treat-\nment -> treatment
        text = re.sub(r"(\w)-\s*\n\s*(\w)", r"\1\2", text)

        # Remove excessive dot leaders, but don't destroy normal punctuation
        text = re.sub(r"\.{4,}", " ", text)

        # Normalize whitespace
        text = re.sub(r"\r", "\n", text)
        text = re.sub(r"\n{3,}", "\n\n", text)
        text = re.sub(r"[ \t]+", " ", text)
        text = text.strip()

        # Keep page if it still has meaningful content
        if len(text) > 50:
            new_doc.page_content = text
            cleaned_docs.append(new_doc)

    return cleaned_docs

In [10]:
cleaned_manual = clean_manual_docs(manual)
print(f"Original pages: {len(manual)}")
print(f"Cleaned pages: {len(cleaned_manual)}")

# inspect the first few pages of the cleaned manual to verify cleaning results and ensure that important content is preserved while unwanted elements are removed.
for i in range(5):
    print(f"\n--- Cleaned Page {i} ---\n")
    print(cleaned_manual[i].page_content[:1000])

Original pages: 4114
Cleaned pages: 4112

--- Cleaned Page 0 ---

Table of Contents
 
Front 
 
Cover 
 
Front Matter 
 
1 - Nutritional Disorders 
 
Chapter 1. Nutrition: General Considerations 
 
Chapter 2. Undernutrition 
 
Chapter 3. Nutritional Support 
 
Chapter 4. Vitamin Deficiency, Dependency & Toxicity 
 
Chapter 5. Mineral Deficiency & Toxicity 
 
Chapter 6. Obesity & the Metabolic Syndrome 
 
2 - Gastrointestinal Disorders 
 
Chapter 7. Approach to the Patient With Upper GI Complaints 
 
Chapter 8. Approach to the Patient With Lower GI Complaints 
 
Chapter 9. Diagnostic & Therapeutic GI Procedures 
 
Chapter 10. GI Bleeding 
 
Chapter 11. Acute Abdomen & Surgical Gastroenterology 
 
Chapter 12. Esophageal & Swallowing Disorders 
 
Chapter 13. Gastritis & Peptic Ulcer Disease 
 
Chapter 14. Bezoars & Foreign Bodies 
 
Chapter 15. Pancreatitis 
 
Chapter 16. Gastroenteritis 
 
Chapter 17. Malabsorption Syndromes 
 
Chapter 18. Irritable Bowel Syndrome 
 
Chapter 19. Inflammat

### Data Chunking

In [13]:
# Chunking the manual into smaller pieces for better processing and retrieval

# The medical manual text was split into overlapping chunks using a recursive character-based text splitter. 
# A chunk size of 800 characters with 150-character overlap was used to preserve semantic context while ensuring chunks remained suitable for embedding and retrieval. 
# Overlapping chunks help prevent loss of important medical information at chunk boundaries and improve retrieval accuracy in the RAG pipeline.

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1024,
    chunk_overlap=128,
    separators=["\n\n", "\n", ".", " ", ""]
)

chunks = splitter.split_documents(cleaned_manual)
print(f"Total chunks: {len(chunks)}\n")
print(f"First chunk:\n {chunks[0].page_content}")



Total chunks: 15933

First chunk:
 Table of Contents
 
Front 
 
Cover 
 
Front Matter 
 
1 - Nutritional Disorders 
 
Chapter 1. Nutrition: General Considerations 
 
Chapter 2. Undernutrition 
 
Chapter 3. Nutritional Support 
 
Chapter 4. Vitamin Deficiency, Dependency & Toxicity 
 
Chapter 5. Mineral Deficiency & Toxicity 
 
Chapter 6. Obesity & the Metabolic Syndrome 
 
2 - Gastrointestinal Disorders 
 
Chapter 7. Approach to the Patient With Upper GI Complaints 
 
Chapter 8. Approach to the Patient With Lower GI Complaints 
 
Chapter 9. Diagnostic & Therapeutic GI Procedures 
 
Chapter 10. GI Bleeding 
 
Chapter 11. Acute Abdomen & Surgical Gastroenterology 
 
Chapter 12. Esophageal & Swallowing Disorders 
 
Chapter 13. Gastritis & Peptic Ulcer Disease 
 
Chapter 14. Bezoars & Foreign Bodies 
 
Chapter 15. Pancreatitis 
 
Chapter 16. Gastroenteritis 
 
Chapter 17. Malabsorption Syndromes 
 
Chapter 18. Irritable Bowel Syndrome 
 
Chapter 19. Inflammatory Bowel Disease 
 
Chapter 20

In [14]:
# Check the overlap between the first few chunks to ensure that the splitting is working as intended and that important context is preserved across chunk boundaries.

num_preview = 5
print(f"Total chunks: {len(chunks)}\n")

for i in range(num_preview - 1):
    a = chunks[i].page_content
    b = chunks[i+1].page_content

    overlap = a[-150:]
    similarity = overlap[:80] in b or overlap[-80:] in b

    print(f"Chunk {i} → {i+1} | Overlap Likely OK: {similarity}")


# Preview the similarity between adjacent chunks to check for excessive overlap or redundancy

chunk_lengths = [len(c.page_content) for c in chunks]

print(f"\nMin chunk: {min(chunk_lengths)}")
print(f"Max chunk: {max(chunk_lengths)}")
print(f"Avg chunk: {sum(chunk_lengths)/len(chunk_lengths)}")

Total chunks: 15933

Chunk 0 → 1 | Overlap Likely OK: True
Chunk 1 → 2 | Overlap Likely OK: True
Chunk 2 → 3 | Overlap Likely OK: False
Chunk 3 → 4 | Overlap Likely OK: False

Min chunk: 26
Max chunk: 1024
Avg chunk: 877.9389317768154


**Chunking and Overlap Observations**

| Insight Category        | Observation                                         | Interpretation                                                                                          | Implication                                                                  |
| ----------------------- | --------------------------------------------------- | ------------------------------------------------------------------------------------------------------- | ---------------------------------------------------------------------------- |
| Total Chunks            | 20,678 chunks generated                             | The document has been split into a large number of manageable segments suitable for vector indexing     | Adequate coverage of the full medical manual for retrieval                   |
| Chunk Size Distribution | Min: 53, Max: 800, Avg: ~695                        | Most chunks are close to the target size, indicating effective chunking; a few very small chunks remain | Overall chunking is well-balanced, but very small chunks may introduce noise |
| Overlap Behaviour       | Chunk 0→1: True, 1→2: True, 2→3: False, 3→4: False  | Overlap is not always exact due to recursive splitting based on natural text boundaries                 | Expected behaviour; semantic continuity is still preserved                   |
| Overlap Effectiveness   | Majority of adjacent chunks show overlap continuity | Overlap mechanism is functioning correctly in most cases                                                | Helps retain context across chunk boundaries for better retrieval            |
| Short Chunks            | Presence of chunks as small as 53 characters        | Likely caused by headings, formatting remnants, or split boundaries                                     | These may reduce retrieval quality and can be filtered out                   |
| Overall Chunk Quality   | Average chunk size is high and consistent           | Indicates that chunks contain meaningful medical content rather than fragmented text                    | Suitable for embedding and FAISS indexing                                    |

The chunking process produced well-sized, context-preserving segments with effective overlap, making the dataset suitable for embedding and retrieval in the RAG pipeline.

### Embedding

In [16]:
# Create embeddings for the chunks using HuggingFaceEmbeddings and store them in a faiss vector database for efficient retrieval during RAG.

embedding_model = HuggingFaceBgeEmbeddings(
    model_name="BAAI/bge-small-en",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

In [17]:
embedding_1 = embedding_model.embed_query(chunks[0].page_content)
embedding_2 = embedding_model.embed_query(chunks[1].page_content)

print("Dimension of the embedding vector ",len(embedding_1))
print(f"Are the dimensions equal? {len(embedding_1) == len(embedding_2)}")

Dimension of the embedding vector  384
Are the dimensions equal? True


**Observations**

- The embedding model produced vectors of dimension 384 for all text chunks, confirming consistent vector representation across the dataset. This ensures compatibility with the FAISS vector index and allows reliable similarity comparison during retrieval.
- The use of normalized embeddings ensures that similarity comparisons (e.g., cosine similarity) are meaningful and consistent, improving the accuracy of retrieved medical context.

### Vector Database and Indexing

- -The generated embeddings were stored in a FAISS vector index to enable fast similarity-based retrieval. For application use, the FAISS index was persisted locally after creation, allowing the system to load precomputed embeddings without recomputing them on each run. This significantly improves performance and ensures a responsive user experience in the deployed medical question-answering application.

- Persisting the FAISS index avoids repeated embedding computation and enables efficient reuse of the vector database, which is essential for real-time applications.

In [ ]:
# Build FAISS vector store from document chunks and their corresponding embeddings.

vectorstore = FAISS.from_documents(
    chunks,
    embedding_model
    )

# save the vector store locally for later use in RAG retrieval & faster reruns
vectorstore.save_local("medical_manual_faiss_index")

In [ ]:
# A set of test queries was run against the vector store to verify that the similarity search is retrieving relevant chunks from the medical manual based on the query.

results = vectorstore.similarity_search("What is the protocol for managing sepsis?", k=3)

for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---\n")
    print(doc.page_content[:500])


In [ ]:
# Reload the index (for inference)

vectorstore = FAISS.load_local(
    "medical_manual_faiss_index",
    embedding_model,
    allow_dangerous_deserialization=True
)


### Retriever

A top-k retrieval strategy (k = 3) was used to balance completeness and precision. Retrieving too few chunks may omit critical medical context, while retrieving too many may introduce irrelevant information and reduce answer quality. Given the chunk size and document structure, k = 3 provides sufficient context for generating accurate and concise medical responses.

In [ ]:
#  set up the retriever to fetch the top 3 most relevant chunks from the vector store based on cosine similarity of the embeddings.

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

### System and User Prompt Template

### Response Function

In [ ]:
def generate_rag_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = qna_system_message + '\n' + user_message

    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

### Fine-tuning

## Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answeres generated to the question from the previous section.

- We are using the same Mistral model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

In [ ]:
groundedness_rater_system_message  = ""

In [ ]:
relevance_rater_system_message = ""

In [ ]:
user_message_template = ""

In [ ]:
def generate_ground_relevance_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=3)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Combine user_prompt and system_message to create the prompt
    prompt = f"""[INST]{qna_system_message}\n
                {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
                [/INST]"""

    response = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    answer =  response["choices"][0]["text"]

    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    response_1 = llm(
            prompt=groundedness_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    response_2 = llm(
            prompt=relevance_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    return response_1['choices'][0]['text'],response_2['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

## Actionable Insights and Business Recommendations

<font size=6 color='blue'>Power Ahead</font>
___